In [ ]:
%pip install groq mysql-connector-python

In [ ]:
import os
import mysql.connector
from groq import Groq

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is not set.")

client = Groq(api_key=GROQ_API_KEY)

MYSQL_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "Praveen@2003",
    "database": "car_rental_system",
}

In [ ]:
SCHEMA = """
users
- user_id
- full_name
- email
- city
- signup_date

cars
- car_id
- brand
- model
- year
- daily_rate
- status

bookings
- booking_id
- user_id
- car_id
- start_date
- end_date
- total_amount
- booking_status

payments
- payment_id
- booking_id
- payment_date
- amount_paid
- payment_method
- payment_status

reviews
- review_id
- booking_id
- user_id
- car_id
- rating
- review_text
- review_date

admin
- admin_id
- admin_name
- email
- role
- created_at
""".strip()

In [ ]:
PROMPT_TEMPLATE = """
You are an expert SQL query generator.

Database Schema:
{schema}

Rules:
- Use only the given schema.
- Do not assume columns.
- Use JOIN when needed.
- Return only SQL query.

User Question:
{question}
""".strip()

def build_prompt(question, schema):
    return PROMPT_TEMPLATE.format(question=question, schema=schema)

In [ ]:

def generate_sql(question, schema):
    prompt = build_prompt(question, schema)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    sql_query = response.choices[0].message.content.strip()

    # Remove code fences if the model returns them
    if sql_query.startswith("```"):
        sql_query = sql_query.replace("```sql", "").replace("```", "").strip()

    return sql_query

In [64]:
# Database function (SELECT only)
def run_select_query(sql_query):
    if not sql_query.strip().lower().startswith("select"):
        raise ValueError("Only SELECT queries are allowed.")

    conn = mysql.connector.connect(**MYSQL_CONFIG)

    try:
        cursor = conn.cursor()
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        columns = [item[0] for item in (cursor.description or [])]
        return columns, rows
    finally:
        conn.close()

In [ ]:

conn = mysql.connector.connect(**MYSQL_CONFIG)

try:
    cursor = conn.cursor()
    cursor.execute("SHOW TABLES;")
    table_rows = cursor.fetchall()
    actual_tables = [row[0] for row in table_rows]

    # Build schema text from real MySQL table columns
    schema_lines = []
    for table_name in actual_tables:
        schema_lines.append(table_name)
        cursor.execute(f"SHOW COLUMNS FROM {table_name};")
        for col in cursor.fetchall():
            schema_lines.append(f"- {col[0]}")
        schema_lines.append("")
finally:
    conn.close()

print("Tables found in MySQL database:")
print(actual_tables)

SCHEMA = "\n".join(schema_lines).strip()
print("\nSchema loaded from MySQL:")
print(SCHEMA)

Tables found in MySQL database:
['admin', 'bookings', 'cars', 'payments', 'reviews', 'users']

Schema loaded from MySQL:
admin
- admin_id
- username
- password
- role
- name
- email

bookings
- booking_id
- user_id
- car_id
- pickup_date
- return_date
- booking_status
- total_amount
- status
- total_price

cars
- car_id
- make
- model
- year
- registration_number
- category
- daily_rate
- status
- available_count
- image_url
- fuel_type
- transmission

payments
- payment_id
- booking_id
- amount
- payment_date
- payment_method
- payment_status

reviews
- review_id
- booking_id
- rating
- comment

users
- user_id
- username
- password
- email
- first_name
- last_name
- phone_number
- role


In [66]:
# Ask one question and generate SQL
question = "Top 5 most booked cars"

sql = generate_sql(question, SCHEMA)
print("Generated SQL:\n")
print(sql)

# Simple table-name validation (basic string check)
sql_lower = sql.lower()
used_tables = []
for table_name in actual_tables:
    if f" {table_name.lower()} " in f" {sql_lower} ":
        used_tables.append(table_name)

print("\nTables referenced from known MySQL tables:")
print(used_tables)

# Warn if SQL references schema tables that are not in MySQL
schema_tables = ["users", "cars", "bookings", "payments", "reviews", "admin"]
missing_tables = []
for table_name in schema_tables:
    if f" {table_name} " in f" {sql_lower} " and table_name not in actual_tables:
        missing_tables.append(table_name)

if missing_tables:
    print("\nWarning: Generated SQL uses tables not present in database")
    print("Missing tables:", missing_tables)

columns, rows = run_select_query(sql)

print("\nResults:\n")
if columns:
    print(" | ".join(columns))
    print("-" * 80)

for row in rows:
    print(" | ".join(str(value) for value in row))

Generated SQL:

SELECT c.car_id, c.make, c.model, COUNT(b.car_id) as total_bookings
FROM cars c
JOIN bookings b ON c.car_id = b.car_id
GROUP BY c.car_id, c.make, c.model
ORDER BY total_bookings DESC
LIMIT 5;

Tables referenced from known MySQL tables:
['bookings', 'cars']

Results:

car_id | make | model | total_bookings
--------------------------------------------------------------------------------
1 | Maruti Suzuki | Swift | 3
2 | Hyundai | Creta | 1
5 | Kia | Seltos | 1
6 | Toyota | Camry | 1
